In [1]:

import logging
from pathlib import Path
from osgeo import gdal
from utils import (
    ImagePreprocessor,
    TileGenerator,
    YOLODetector,
    DetectionPostprocessor,
    QanatClusterer
)

import numpy as np

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

# ============================================
# CONFIGURATION
# ============================================
rgb_path = "datasets/AFG2_D3C1219-merged.tif"
model_path = "/home/nazar/2024_JAS_qanats/ultralytics/runs/detect/train/weights/best.pt"
tile_size = 1024
output_dir = "results/"

# Create directories
Path(output_dir).mkdir(exist_ok=True)
Path("temp").mkdir(exist_ok=True)

# ============================================
# STEP 1-4: Preprocess Image
# ============================================
preprocessor = ImagePreprocessor()

# 1. Grayscale
gray_path = preprocessor.convert_to_grayscale_8bit(
    rgb_path,
    "temp/image_gray.tif"
)

# 2. Reproject to EPSG:4326
proj_path = preprocessor.reproject_to_4326(
    gray_path,
    "temp/image_4326.tif"
)

# 3. Clip/pad to regular size (automatic!)
clip_path = preprocessor.clip_or_pad_to_regular_size(
    proj_path,
    "temp/image_clipped.tif",
    tile_size=tile_size,
    use_padding=False  # Clip mode
)

# 4. Extract reference geotransform
ref_ds = gdal.Open(clip_path)
reference_geotransform = ref_ds.GetGeoTransform()
clip_width = ref_ds.RasterXSize
clip_height = ref_ds.RasterYSize
ref_ds = None

print(f"Reference image: {clip_width}×{clip_height}")
print(f"Reference geotransform: {reference_geotransform}")

# ============================================
# STEP 5: Generate Tiles
# ============================================
tile_gen = TileGenerator(tile_size=tile_size, overlap=512)
tiles = tile_gen.generate_tiles_from_geotiff(
    clip_path,
    "temp/tiles/"
)

print(f"Generated {len(tiles)} tiles")

# ============================================
# STEP 6: YOLO Detection
# ============================================
detector = YOLODetector(
    model_path=model_path,
    conf_threshold=0.1,  # Initial threshold
    iou_threshold=0.45,
    image_size=tile_size
)

detections = detector.detect_tiles(
    tiles,
    batch_size=16,
    save_results=True,
    output_dir="temp/detections/"
)

total_dets = sum(d['num_detections'] for d in detections)
print(f"Total detections: {total_dets}")

# ============================================
# STEP 7: Merge Tile Detections
# ============================================
postprocessor = DetectionPostprocessor(iou_threshold=0.5)
merged = postprocessor.merge_tile_detections(
    detections,
    image_shape=(clip_height, clip_width)
)

print(f"Merged detections: {merged['num_detections']}")
print(f"Classes: {dict(zip(*np.unique(merged['class_names'], return_counts=True)))}")

if merged['num_detections'] == 0:
    print("ERROR: No detections found!")
    exit(1)

# ============================================
# STEP 8: Process with Original Pipeline
# ============================================
clusterer = QanatClusterer(
    eps=0.0008,
    min_samples=4,
    min_confidence_qanat=0.1,
    min_confidence_qanat_pair=0.2,
    overlap_threshold=0.7  # 80% overlap required
)

# This runs the COMPLETE original pipeline:
# - Boxes to GeoDataFrame
# - Dissolve overlapping qanat bboxes
# - Spatial join with qanat_pairs
# - Confidence filtering
# - DBSCAN clustering
# - Remove outliers


2025-11-10 20:36:10,409 - INFO - Converting to grayscale 8-bit: datasets/AFG2_D3C1219-merged.tif
2025-11-10 20:36:10,657 - INFO - Input has 1 band(s)
2025-11-10 20:36:10,658 - INFO - Input is already grayscale, converting to 8-bit
2025-11-10 20:36:10,736 - INFO - Output array shape: (7424, 7424), dtype: uint8
2025-11-10 20:36:10,862 - INFO - Grayscale conversion completed: temp/image_gray.tif
2025-11-10 20:36:10,865 - INFO - Reprojecting to EPSG:4326: temp/image_gray.tif
2025-11-10 20:36:13,218 - INFO - Reprojection completed: temp/image_4326.tif
2025-11-10 20:36:13,219 - INFO - Processing to regular size with tile_size=1024, padding=False
2025-11-10 20:36:13,227 - INFO - Original size: 8200x7023
2025-11-10 20:36:13,228 - INFO - Clipped size: 8192x6144
2025-11-10 20:36:13,327 - INFO - Processed image saved to temp/image_clipped.tif
2025-11-10 20:36:13,329 - INFO - Initialized TileGenerator: tile_size=1024, overlap=512, stride=512
2025-11-10 20:36:13,330 - INFO - Generating georeference

Reference image: 8192×6144
Reference geotransform: (65.21296243672946, 8.48674209043992e-06, 0.0, 31.561581684119933, 0.0, -8.48674209043992e-06)


2025-11-10 20:36:13,725 - INFO - Generated 165 georeferenced tiles
2025-11-10 20:36:13,729 - INFO - Saved tile metadata to temp/tiles/tile_metadata.json


Generated 165 tiles


2025-11-10 20:36:14,545 - INFO - Successfully loaded YOLO model from /home/nazar/2024_JAS_qanats/ultralytics/runs/detect/train/weights/best.pt
2025-11-10 20:36:14,546 - INFO - Initialized YOLODetector on cuda
2025-11-10 20:36:14,546 - INFO - Model: /home/nazar/2024_JAS_qanats/ultralytics/runs/detect/train/weights/best.pt
2025-11-10 20:36:14,547 - INFO - Conf threshold: 0.1, IOU threshold: 0.45
2025-11-10 20:36:14,547 - INFO - Image size: 1024
2025-11-10 20:36:14,548 - INFO - Running batch detection on 165 images
2025-11-10 20:36:23,795 - INFO - Completed batch detection: 165 results
2025-11-10 20:36:23,832 - INFO - Merged 3709 detections to 1181 after NMS
2025-11-10 20:36:23,834 - INFO - QanatClusterer initialized: eps=0.0008, min_samples=4, overlap_threshold=70.0%


Total detections: 3709
Merged detections: 1181
Classes: {'qanat': 600, 'qanat_pair': 581}


In [2]:
gdf = clusterer.process_detections(
    merged,
    reference_geotransform=reference_geotransform,
    crs="EPSG:4326"
)

# Export with centroids (default)
clusterer.export_results(
    gdf, 
    "results/qanat_detections.gpkg",
    driver="GPKG",
    include_centroids=True  # Default is True
)


2025-11-10 20:36:23,839 - INFO - ============================================================
2025-11-10 20:36:23,840 - INFO - QANAT DETECTION PROCESSING PIPELINE
2025-11-10 20:36:23,841 - INFO - ============================================================
2025-11-10 20:36:23,841 - INFO - Step 1: Converting detections to GeoDataFrame
2025-11-10 20:36:23,873 - INFO - Created GeoDataFrame with 1181 detections
2025-11-10 20:36:23,874 - INFO - Total detections: 1181
2025-11-10 20:36:23,874 - INFO - Classes: {'qanat': 600, 'qanat_pair': 581}
2025-11-10 20:36:23,875 - INFO - 
Step 2: Separating qanat and qanat_pair classes
2025-11-10 20:36:23,877 - INFO - Qanat: 600, Qanat_pair: 581
2025-11-10 20:36:23,877 - INFO - 
Step 3: Dissolving overlapping qanat bboxes
2025-11-10 20:36:23,878 - INFO - Dissolving 600 qanat bboxes (overlap threshold: 70.0%)
2025-11-10 20:36:23,971 - INFO - Dissolved to 587 bboxes (from 600)
2025-11-10 20:36:23,972 - INFO - Merged 13 boxes with >70.0% overlap
2025-11-10 

'results/qanat_detections.gpkg'

In [3]:
import geopandas as gpd
from shapely import wkt  

# Load main file
gdf = gpd.read_file("results/qanat_detections.gpkg")

print(gdf.geometry)  # Bounding boxes (Polygon)
print(gdf.centroid_x, gdf.centroid_y)  # Centroid coordinates
print(gdf.centroid_wkt)  # Centroid as WKT text

# Convert WKT to geometry if needed
gdf['centroid_geom'] = gdf['centroid_wkt'].apply(wkt.loads)

# Load centroids file
gdf_centroids = gpd.read_file("results/qanat_detections_centroids.gpkg")

print(gdf_centroids.geometry)  # Centroids (Point)
print(gdf_centroids[['bbox_minx', 'bbox_miny', 'bbox_maxx', 'bbox_maxy']])
print(gdf_centroids.bbox_wkt)  # Bbox as WKT text

# Convert WKT to geometry if needed
gdf_centroids['bbox_geom'] = gdf_centroids['bbox_wkt'].apply(wkt.loads)

0      POLYGON ((65.21482 31.52604, 65.21482 31.52616...
1      POLYGON ((65.24301 31.52588, 65.24301 31.52603...
2      POLYGON ((65.25303 31.52496, 65.25303 31.52509...
3      POLYGON ((65.21503 31.52639, 65.21503 31.52652...
4      POLYGON ((65.24332 31.52646, 65.24332 31.52661...
                             ...                        
520    POLYGON ((65.25875 31.51242, 65.25875 31.51252...
521    POLYGON ((65.26452 31.5164, 65.26452 31.51654,...
522    POLYGON ((65.22377 31.52435, 65.22377 31.52452...
523    POLYGON ((65.22335 31.52417, 65.22335 31.52432...
524    POLYGON ((65.24699 31.52354, 65.24699 31.52363...
Name: geometry, Length: 525, dtype: geometry
0      65.214755
1      65.242939
2      65.252968
3      65.214965
4      65.243244
         ...    
520    65.258702
521    65.264445
522    65.223690
523    65.223262
524    65.246931
Name: centroid_x, Length: 525, dtype: float64 0      31.526103
1      31.525956
2      31.525023
3      31.526455
4      31.526538
         .

In [4]:


# ============================================
# STEP 10: Visualize (Optional)
# ============================================
from utils import ResultVisualizer

visualizer = ResultVisualizer()
map_obj = visualizer.visualize_clusters(
    raster_path=clip_path,
    gdf=gdf_centroids,
    cluster_column="cluster",
    output_html=f"{output_dir}/results_map.html"
)

print(f"Map saved to: {output_dir}/results_map.html")

2025-11-10 20:36:26,550 - INFO - Saved clustered map to results/results_map.html
2025-11-10 20:36:26,550 - INFO - Visualized 20 qanat clusters with 525 total shafts


Map saved to: results//results_map.html


from utils.visualization_export import ResultsExporter, ResultsVisualizer
from utils.evaluation import ResultsEvaluator


# Evaluate
evaluator = ResultsEvaluator("evaluation/")
results = evaluator.run_evaluation_pipeline(
    detected_file="results/qanat_detections.gpkg",
    reference_file="datasets/AFG2_D3C1219-reference.gpkg",
    buffer_distance=0.00004
)
evaluator.print_metrics(results['metrics'])

# Export results
exporter = ResultsExporter("exports/")
exports = exporter.to_all_formats("results/qanat_detections.gpkg")

# Visualize
visualizer = ResultsVisualizer()
visualizer.print_summary("results/qanat_detections.gpkg")
m = visualizer.create_interactive_map("results/qanat_detections.gpkg")


In [5]:
from utils.evaluation import ResultsEvaluator

evaluator = ResultsEvaluator("evaluation/")

results = evaluator.run_evaluation_pipeline(
    detected_file="results/qanat_detections.gpkg",
    reference_file="datasets/AFG2_D3C1219-reference.gpkg",
    buffer_distance=0.00004,
    save_comparison_files=True,
    export_formats=['gpkg', 'geojson', 'csv']
)

# Access files
tp_file = results['comparison_files']['gpkg']['true_positives']
fp_file = results['comparison_files']['gpkg']['false_positives']
fn_file = results['comparison_files']['gpkg']['false_negatives']

print(f"TP: {tp_file}")
print(f"FP: {fp_file}")
print(f"FN: {fn_file}")

2025-11-10 20:36:26,586 - INFO - Initialized evaluator: evaluation/
2025-11-10 20:36:26,586 - INFO - 
2025-11-10 20:36:26,586 - INFO - STARTING EVALUATION PIPELINE
2025-11-10 20:36:26,587 - INFO - ================================================================================
2025-11-10 20:36:26,587 - INFO - 
[Step 1] Creating buffers...
2025-11-10 20:36:26,587 - INFO - Creating buffers with distance 4e-05...
2025-11-10 20:36:26,593 - INFO - Loaded 525 features
2025-11-10 20:36:26,754 - INFO - ✓ Buffers created: evaluation/detected_buffer.gpkg
2025-11-10 20:36:26,755 - INFO - Creating buffers with distance 4e-05...
2025-11-10 20:36:26,760 - INFO - Loaded 821 features
2025-11-10 20:36:26,869 - INFO - ✓ Buffers created: evaluation/reference_buffer.gpkg
2025-11-10 20:36:26,870 - INFO - 
[Step 2] Spatial join (detected vs reference)...
2025-11-10 20:36:26,871 - INFO - Spatial join: detected features with reference buffers...
2025-11-10 20:36:26,871 - INFO - Using centroids file: results/q

TP: evaluation/true_positives.gpkg
FP: evaluation/false_positives.gpkg
FN: evaluation/false_negatives.gpkg


In [6]:
evaluator.print_metrics(results['metrics'])



EVALUATION METRICS

📊 Confusion Matrix:
   • True Positives (TP):   473
   • False Positives (FP):  56
   • False Negatives (FN):  290

🎯 Performance:
   • Precision: 0.8941 (89.41%)
   • Recall:    0.6199 (61.99%)
   • F1-Score:  0.7322 (73.22%)

📈 Summary:
   • Total Detected:   529
   • Total Reference:  763

